In [ ]:
import os
import sys
import time
import re
from pathlib import Path

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 0. Configuration de l'environnement (Adapté à Jupyter & .env du projet)

In [ ]:
from api_config import configure_google_api_key

# Configurer la clé API Google (charge depuis Colab Secrets ou .env local)
GOOGLE_API_KEY = configure_google_api_key()

# 1. Initialisation de la base de données et des Embeddings (Recommandé dans un Cell séparé)


In [ ]:
print("🔄 Initialisation des instances et chargement de Chroma...")
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-2",
    task_type="retrieval_document"
)
vectorstore_disk = Chroma(
    persist_directory="./chroma_minecraft_db",
    embedding_function=gemini_embeddings
)

# Liste des modèles de secours en cas d'erreur 503 de surcharge
MODELS_TO_TRY = ["gemini-2.5-flash", "gemini-1.5-flash", "gemini-1.5-pro"]

# 2. Définition des Prompts (Recommandé dans un Cell séparé)


In [ ]:
# Prompt conversationnel classique
original_template = """Tu es un expert du jeu Minecraft et un professeur de l'UE LO17.
Écris un paragraphe de réponse fictif mais extrêmement précis et rigoureux, rédigé en français simple, pour répondre à la question ci-dessous.
N'utilise aucun formatage markdown (pas d'astérisques **, pas de backticks `, pas de listes). Écris uniquement du texte brut.
Ce paragraphe servira de document d'exemple pour effectuer une recherche sémantique (similarité vectorielle). 

Question: {question}
Document fictif (réponse technique directe) :"""

# Prompt dense optimisé pour la consommation de tokens
dense_template = """Tu es un expert technique du jeu Minecraft.
Écris un condensé d'assertions techniques brutes et de mots-clés sémantiques indispensables pour répondre à la question suivante.
Bannis la grammaire superflue, les phrases d'introduction, les transitions et les formules de politesse.
Produis uniquement des faits ultra-denses sous forme de texte brut.
Rédige impérativement en français (le même langage que la question).
Maximum 25 mots. Aucun formatage markdown.

Question: {question}
Assertions sémantiques brutes :"""

original_prompt = PromptTemplate.from_template(original_template)
dense_prompt = PromptTemplate.from_template(dense_template)

# 3. Fonction génératrice HyDE robuste (Recommandé dans un Cell séparé)


In [ ]:
def generate_hyde_document(question, mode="dense"):
    """
    Génère un document hypothétique avec gestion des erreurs,
    dégradation logicielle et option d'économie de tokens.
    """
    selected_prompt = dense_prompt if mode == "dense" else original_prompt
    temperature = 0.1 if mode == "dense" else 0.3
    
    hypothetical_doc = None
    selected_model_name = None
    
    for model_name in MODELS_TO_TRY:
        for attempt in range(3):
            try:
                llm = ChatGoogleGenerativeAI(
                    model=model_name, 
                    temperature=temperature
                )
                hyde_chain = selected_prompt | llm | StrOutputParser()
                hypothetical_doc = hyde_chain.invoke({"question": question})
                selected_model_name = model_name
                break
            except Exception as e:
                err_msg = str(e)
                if "503" in err_msg or "UNAVAILABLE" in err_msg or "high demand" in err_msg:
                    wait_time = (attempt + 1) * 3
                    print(f"   ⚠️ Modèle {model_name} saturé. Tentative {attempt+1}/3. Pause de {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"   ❌ Erreur avec {model_name} : {e}")
                    break
        if hypothetical_doc:
            break
            
    if not hypothetical_doc:
        raise RuntimeError("Échec critique : Impossible de joindre les API Gemini.")
        
    # Nettoyage des formats Markdown résiduels
    cleaned_doc = re.sub(r'[\*`_#]', '', hypothetical_doc).strip()
    return cleaned_doc, selected_model_name

# 4. Exécution de l'expérience comparative (Recommandé dans un Cell séparé)


In [ ]:
question = "Quel est l'ingrédient de base indispensable pour l'alchimie ?"
print(f"\n❓ Question d'origine : '{question}'")

# Générer le HyDE classique
print("\n🤖 [Mode Standard] Génération du document hypothétique classique...")
doc_orig, model_orig = generate_hyde_document(question, mode="original")
word_count_orig = len(doc_orig.split())
print(f" ✅ Succès ({model_orig}) | Taille : {word_count_orig} mots.")
print(f"📄 Passage :\n\"\"\"\n{doc_orig}\n\"\"\"")

# Générer le HyDE ultra-dense (économe en tokens)
print("\n🤖 [Mode Dense] Génération de l'empreinte sémantique compressée (Économie de tokens)...")
doc_dense, model_dense = generate_hyde_document(question, mode="dense")
word_count_dense = len(doc_dense.split())
print(f" ✅ Succès ({model_dense}) | Taille : {word_count_dense} mots (Économie de ~{100 - int(word_count_dense/word_count_orig*100)}% de tokens !).")
print(f"📄 Passage :\n\"\"\"\n{doc_dense}\n\"\"\"")

# 5. Exécution de la recherche vectorielle


In [ ]:
print("\n🔍 Lancement de la recherche sémantique standard (Question pure)...")
results_standard = vectorstore_disk.similarity_search(question, k=4)

print("\n🔍 Lancement de la recherche HyDE Standard...")
results_hyde_orig = vectorstore_disk.similarity_search(doc_orig, k=4)

print("\n🔍 Lancement de la recherche HyDE Dense (Économe)...")
results_hyde_dense = vectorstore_disk.similarity_search(doc_dense, k=4)

# 6. Présentation des résultats comparatifs

In [ ]:
print("\n" + "="*80)
print("🏆 COMPARAISON ACADÉMIQUE DES MODES DE RECHERCHE (HyDE Standard vs HyDE Dense)")
print("="*80)

print("\n🔵 [1] RECHERCHE STANDARD (Question pure)")
for i, doc in enumerate(results_standard):
    text_clean = doc.page_content[:200].replace('\n', ' ')
    print(f"[{i+1}] Source: {doc.metadata.get('source')} | {text_clean}...")

print("\n🟡 [2] RECHERCHE HYDE CLASSIQUE (Passage romancé - {0} mots)".format(word_count_orig))
for i, doc in enumerate(results_hyde_orig):
    text_clean = doc.page_content[:200].replace('\n', ' ')
    print(f"[{i+1}] Source: {doc.metadata.get('source')} | {text_clean}...")

print("\n🟢 [3] RECHERCHE HYDE ULTRA-DENSE (Économe en Tokens - {0} mots)".format(word_count_dense))
for i, doc in enumerate(results_hyde_dense):
    text_clean = doc.page_content[:200].replace('\n', ' ')
    print(f"[{i+1}] Source: {doc.metadata.get('source')} | {text_clean}...")